# T-FOLEY: sınıf ve RMS koşullu inference

Metin kullanılmaz. Sınıf `Footstep`, seed 42, 100 adım ve dört saniyelik üç pulse RMS referansı kullanılır. Pulse referansı gerçek ayak sesi değildir. İlk inference, temporal kontrol başarısı veya ses kalitesi benchmark'ı değildir.

Önkoşullar: Git, temel requirements, PyTorch ve `requirements-tfoley.txt`. Mevcut CUDA PyTorch kurulumu korunur. İlk çalıştırmada resmî kod ve yaklaşık 549 MB checkpoint indirilir; revizyon ve ZIP checksum doğrulanır. `device=auto` kullanılabilir CUDA GPU'yu seçer, yoksa CPU. CPU koşumu uzun sürebilir.


In [ ]:
from pathlib import Path
import sys, subprocess, json
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from IPython.display import Audio, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/baselines/tfoley_inference.py").exists()
import torch
print({"torch": torch.__version__, "cuda_available": torch.cuda.is_available()})


## Koşullama ve üretim

RMS çıkarımı frame 512/hop 128 ve elliptic smoothing ile resmî modele uygun yapılır. Guidance 3. RFF embedding'deki sabit CUDA yönlendirmesi giriş tensörünün device'ına uyarlanır. Model yapısı ve EMA ağırlıkları değişmez. Bu koşum önceki `first_inference` sonuçlarını koruyarak `local_inference` dosyalarına yazar.


In [ ]:
command = [sys.executable, str(ROOT / "src/baselines/tfoley_inference.py"),
           "--prepare", "--device", "auto", "--steps", "100", "--seed", "42",
           "--class-name", "Footstep", "--output-name", "local_inference"]
# Kendi RMS referans WAV'ınız için ekleyin:
# command += ["--target-audio", str(ROOT / "data/raw/reference.wav")]
subprocess.run(command, cwd=ROOT, check=True)


## Çıktı ve koşum kaydı

FLOAT WAV yerel `results/audio/tfoley/` altında saklanır. RMS eğrisi enerji zarfını gösterir; perceptual loudness değildir. Tek örnek ve dinleme, FAD veya çoklu kontrol ayrıştırma değerlendirmesinin yerine geçmez.


Model çıktısı ±1 aralığını aşabilir. FLOAT WAV kırpılmadan saklanır; iki player aynı playback gain ile dinletir. Ölçümler normalize edilmemiş ses üzerinde yapılır.

In [ ]:
run_path = ROOT / "experiments/baselines/tfoley/local_inference_run.json"
report = json.loads(run_path.read_text(encoding="utf-8"))
report


In [ ]:
folder = ROOT / "results/audio/tfoley"
y, sr = sf.read(folder / "local_inference.wav")
reference, _ = sf.read(folder / "local_inference_reference.wav")
playback_divisor = max(float(np.max(np.abs(reference))), float(np.max(np.abs(y))), np.finfo(float).eps)
display(Markdown("**Sentetik RMS referansı — gerçek ayak sesi değil**"))
display(Audio(reference / playback_divisor, rate=sr, normalize=False))
display(Markdown("**T-FOLEY çıktısı — Footstep sınıfı**"))
display(Audio(y / playback_divisor, rate=sr, normalize=False))
fig, ax = plt.subplots(figsize=(10, 3))
for signal, label in [(reference, "Reference RMS"), (y, "Generated RMS")]:
    rms = librosa.feature.rms(y=signal.astype(np.float32), frame_length=512, hop_length=128)[0]
    ax.plot(librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=128), rms, label=label)
ax.set(xlabel="Time (s)", ylabel="Unweighted RMS")
ax.legend()
plt.show()
